# 리포트 54 — 그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다

> ### 한 일
> **명목과 경험 사이의 배율을 만드는 항을 대조군 두 종으로 하나씩 꺼 확정하고, 그 배율이 형상에 얼마나 의존하는지를 창 폭을 바꿔 재었다.**

### 결과
1. CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 slow-time Hann 창으로 도플러축 셀을 묶는다 — Hann 을 rect 창으로 바꾸면 5G 배율이 0.96 [^1] 로 내려온다.
2. 거리축 항까지 끄면 1.02 [^2] 로 눈금이 1 로 돌아온다 — 두 항이 배율의 전부다.
3. 형상이 배율을 정한다 — 같은 파형이 운용 창에서 1.52 [^3]배, 넓은 창(256 [^4] 빈)에서 47.70 [^5]배다.
4. 그래서 교정표는 형상마다 다시 잰다. `check_detector_config()`(`src/passive_process.py:383`)가 거리창과 0-도플러 마스크 두 조건을 강제한다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 대조군 2종 | slow-time Hann 제거(도플러축) · 백색화 정합필터(거리축). 하나씩 끄고 배율이 어디로 가는지를 본다 |
| 사다리 읽기 | 이상적 백색 맵 → 잡음 맵 → 항을 하나씩 뺀 대조군 → 전체 사슬 순서로 읽으면 각 항의 몫이 갈린다 |
| 형상 의존 | 운용 창과 넓은 창(256 [^4] 빈)에서 같은 파형을 다시 재 배율의 형상 의존을 크기로 적는다 |
| 두 형상의 쓰임 | 운용 형상의 표는 챔버 기하가 읽고, 자유공간 기하는 `src/freespace_detect.py:711` 이 자기 형상에서 다시 잰다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json` |
| 소요 | CFAR 측정이 2717 s [^6] (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 53 «운용 형상에서 경험 Pfa 를 재니 명목값의…»](53_cfar-calib.ipynb) | 운용 형상에서 잰 경험 Pfa 와 교정표 |

---

## 원인 — 셀 상관

CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 slow-time Hann 창으로 도플러축 셀을 묶고, 정합필터로 거리축 셀을 묶는다. 대조군이 그 두 항을 원인으로 확정한다(5G NR, 명목 1e-4).

| 조건 | 경험/명목 |
|---|---|
| 이상적 백색 맵 | 0.997 [^7] |
| 잡음 맵 (Hann + 정합필터) | 1.25 [^8] |
|   └ Hann 제거 (rect 창) | 0.96 [^1] |
|   └ 백색화 정합필터 (거리축 평탄) | 1.25 [^9] |
|   └ 둘 다 제거 | 1.02 [^2] |
| 전체 사슬 (직접파 + ECA) | 1.52 [^3] |

![f5_cause](../outputs/figures/report04_f5_cause.png)

**그림 1.** 명목과 경험 사이의 배율을 만드는 것은 무엇인가?

## 셀 상관은 어느 검출기에나 있다 — 그래서 대조군이 필요하다

«상관이 있다» 만으로는 원인 지목이 서지 않는다. 항을 하나씩 꺼서 눈금이 1 로 돌아오는 것을 보여야 확정된다. Hann 을 rect 로 바꾸면 0.96 [^1], 백색화 정합필터까지 끄면 1.02 [^2] 다.

그 사다리가 위 표이고, 마지막 줄과 첫 줄 사이의 간격이 곧 두 항의 몫이다.

## 형상 규약 — 교정표가 성립하는 조건

거리창은 ECA 탭 안에 두고, 0-도플러 행 1 [^10]개를 마스킹한다. `check_detector_config()`(`src/passive_process.py:383`)가 두 조건을 검사한다.

| 파형 | 운용 창 | 넓은 창 |
|---|---|---|
| WiFi | 1.53 [^11]배 | 41.1 [^12]배 |
| LTE | 2.66 [^13]배 | 58.8 [^14]배 |
| 5G | 1.52 [^3]배 | 47.7 [^5]배 |

창을 256 [^4] 빈으로 넓히면 배율이 두 자릿수가 된다. 교정표는 운용 창 형상에서 측정한 값이다.

## 어느 형상의 교정표가 어디에 쓰이나

| 형상 | 무엇을 재나 | 재는 코드 | 그 값을 읽는 곳 |
|---|---|---|---|
| 운용 형상 — CPI 프레임 48 [^15] · `g2x2_t6x6` · 마스크 1 [^10]빈 · 운용 거리창 | 이 부의 교정표 | `benchmark/verify_cfar.py` | `src/experiment_detection.py:358` · `src/experiment_x410.py:175` |
| 자유공간 형상 — 모드별 프레임 수 · 자유공간 거리창 · 0-도플러 가드 | 자유공간 명목 Pfa | `src/freespace_detect.py:711` | `src/experiment_freespace_range.py:206` |

[편 58 «자유공간 형상에서 문턱을 다시 재니 세 밴드가…»](58_shared-threshold.ipynb) 가 싣는 명목 Pfa 는 둘째 줄에서 나온 수다 — 이 부의 교정표와 형상이 달라 값도 다르다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 훈련셀에서도 0-도플러 행을 빼는 CFAR 변형을 만든다 | 넓은 창에서 마스크 폭 3 이 만드는 배율 0.65 [^16]배가 마스크 폭과 분리된다 | `src/passive_process.py:352` |
| 표적모형 민감도를 이 배율 위에서 다시 푼다 | 세 표적모형의 절대 소요이득이 경험 Pfa 위에 선다 — CA-CFAR 문턱은 세 팔에 같은 오프셋을 주므로 모형 간 차이는 문턱 규약에 불변이다 | [편 65 «평판·큐브·우리 격자를 같은 동작점에서 갈아끼…»](65_target-model-swap.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 16개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/verify_cfar.json` | `control_rect_window_NR100.op.rows[89].ratio` | 0.9619 |
| [^2] | `outputs/verify_cfar.json` | `control_whitened_mf_rect_NR100.op.rows[89].ratio` | 1.02 |
| [^3] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].ratio` | 1.521 |
| [^4] | `outputs/verify_cfar.json` | `meta.n_range_wide` | 256 |
| [^5] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.wide.rows[89].ratio` | 47.7 |
| [^6] | `outputs/verify_cfar.json` | `meta.runtime_s` | 2717 |
| [^7] | `outputs/verify_cfar.json` | `white.48x24.rows[89].ratio` | 0.9966 |
| [^8] | `outputs/verify_cfar.json` | `chain.NR100.noise.op.rows[89].ratio` | 1.246 |
| [^9] | `outputs/verify_cfar.json` | `control_whitened_mf_NR100.op.rows[89].ratio` | 1.246 |
| [^10] | `outputs/verify_cfar.json` | `meta.zd_mask_operational` | 1 |
| [^11] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.op.rows[89].ratio` | 1.531 |
| [^12] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.wide.rows[89].ratio` | 41.14 |
| [^13] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.op.rows[89].ratio` | 2.663 |
| [^14] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.wide.rows[89].ratio` | 58.8 |
| [^15] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^16] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.wide.rows[90].ratio` | 0.6538 |